In [ ]:
from pathlib import Path
import pandas as pd
import torch

INPUT_ROOT = Path("/kaggle/input")

# Support both Kaggle input layouts.
data_candidates = [
    INPUT_ROOT / "competitions/rsna-knee-abnormality-detection",
    INPUT_ROOT / "rsna-knee-abnormality-detection",
]

DATA_DIR = next(
    (path for path in data_candidates if (path / "test.csv").is_file()),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError("Competition input not found. Check Add Input.")

# Search only notebook-output locations, not the MRI dataset.
model_candidates = list(
    (INPUT_ROOT / "notebooks").glob("*/*/kneesignal_resnet18_v03_fold0.pt")
)
model_candidates += list(
    INPUT_ROOT.glob("*/kneesignal_resnet18_v03_fold0.pt")
)

model_candidates = sorted(set(model_candidates))

if len(model_candidates) != 1:
    raise RuntimeError(
        f"Found {len(model_candidates)} model files; expected exactly one. "
        "Check that baseline Version #4 is attached."
    )

MODEL_PATH = model_candidates[0]

test = pd.read_csv(DATA_DIR / "test.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")

print("Test studies:", len(test))
print("Test series:", len(test_series))
print("Model path:", MODEL_PATH)
print("Model size (MB):", round(MODEL_PATH.stat().st_size / 1_000_000, 1))
print("GPU available:", torch.cuda.is_available())

In [ ]:
from torch import nn
from torchvision.models import resnet18

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]


class KneeMRIModel(nn.Module):
    def __init__(self, num_targets=12):
        super().__init__()

        backbone = resnet18(weights=None)
        self.image_encoder = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        self.metadata_encoder = nn.Sequential(
            nn.Linear(5, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
        )

        self.attention = nn.Sequential(
            nn.Linear(544, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        self.classifier = nn.Linear(544, num_targets)

    def forward(self, images, metadata, series_mask):
        images = images.float()
        metadata = metadata.float()
        series_mask = series_mask.float()

        batch_size, num_series, num_slices, _, height, width = images.shape

        x = images.reshape(
            batch_size * num_series * num_slices, 1, height, width
        )

        # Process small groups of slices to limit GPU memory usage.
        features = []
        for chunk in x.split(24):
            rgb = chunk.repeat(1, 3, 1, 1)
            features.append(self.image_encoder(rgb).flatten(1))

        image_features = torch.cat(features).reshape(
            batch_size, num_series, num_slices, 512
        ).mean(dim=2)

        metadata_features = self.metadata_encoder(metadata)
        series_features = torch.cat(
            [image_features, metadata_features], dim=-1
        )

        attention_scores = self.attention(series_features).squeeze(-1)
        attention_scores = attention_scores.masked_fill(
            series_mask == 0, -1e4
        )
        attention_weights = attention_scores.softmax(dim=1)

        study_features = (
            series_features * attention_weights.unsqueeze(-1)
        ).sum(dim=1)

        return self.classifier(study_features)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = KneeMRIModel(num_targets=len(TARGETS))

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=True,
)

model.load_state_dict(checkpoint, strict=True)
model = model.to(device)
model.eval()

print("Trained weights loaded successfully")
print("Device:", device)
print("Output targets:", len(TARGETS))

In [ ]:
import numpy as np
import pydicom
import torch.nn.functional as F
from tqdm.auto import tqdm

load_counts = {"unreadable_slices": 0, "empty_series": 0}


def load_slice(path, image_size=224):
    dcm = pydicom.dcmread(path)
    image = dcm.pixel_array.astype(np.float32)

    low, high = np.percentile(image, [1, 99])
    image = np.clip(
        (image - low) / max(high - low, 1e-6), 0, 1
    ).astype(np.float32)

    tensor = torch.from_numpy(image)[None, None]
    return F.interpolate(
        tensor,
        size=(image_size, image_size),
        mode="bilinear",
        align_corners=False,
    )[0]


def load_series(folder, num_slices=24):
    files = sorted(folder.glob("*.dcm"))
    ordered = []

    for fallback, path in enumerate(files):
        try:
            header = pydicom.dcmread(path, stop_before_pixels=True)
            position = int(getattr(header, "InstanceNumber", fallback))
        except Exception:
            position = fallback
        ordered.append((position, path))

    files = [path for _, path in sorted(ordered)]
    slices = []

    if files:
        indices = np.linspace(
            0, len(files) - 1, num_slices
        ).round().astype(int)

        for index in indices:
            try:
                slices.append(load_slice(files[index]))
            except Exception:
                load_counts["unreadable_slices"] += 1

    if not slices:
        load_counts["empty_series"] += 1
        return torch.zeros(num_slices, 1, 224, 224)

    while len(slices) < num_slices:
        slices.append(slices[-1].clone())

    return torch.stack(slices).float()


def load_test_study(study_uid):
    selected = test_series[
        test_series["StudyInstanceUID"] == study_uid
    ].copy()

    if selected.empty:
        raise ValueError("A test study has no listed series.")

    selected["_rank"] = (
        2 * selected["Fluid_Sensitive"].fillna(0)
        + selected["Fat_Suppression"].fillna(0)
    )
    selected = selected.sort_values(
        ["Anatomical_Plane", "_rank"],
        ascending=[True, False],
    )

    first = selected.groupby(
        "Anatomical_Plane", dropna=False
    ).head(1)
    remaining = selected.drop(first.index).sort_values(
        "_rank", ascending=False
    )
    chosen = pd.concat([first, remaining]).head(5)

    images, metadata, mask = [], [], []

    for _, row in chosen.iterrows():
        folder = (
            DATA_DIR / "test_series"
            / str(study_uid) / str(row["SeriesInstanceUID"])
        )
        images.append(load_series(folder))

        plane = row["Anatomical_Plane"]
        metadata.append([
            float(plane == "Sagittal"),
            float(plane == "Coronal"),
            float(plane == "Axial"),
            float(row["Fluid_Sensitive"]),
            float(row["Fat_Suppression"]),
        ])
        mask.append(1.0)

    while len(images) < 5:
        images.append(torch.zeros(24, 1, 224, 224))
        metadata.append([0.0] * 5)
        mask.append(0.0)

    return (
        torch.stack(images).unsqueeze(0),
        torch.tensor(metadata, dtype=torch.float32).unsqueeze(0),
        torch.tensor(mask, dtype=torch.float32).unsqueeze(0),
    )


predictions = []
model.eval()

with torch.inference_mode():
    for study_uid in tqdm(
        test["StudyInstanceUID"], desc="Predicting test studies"
    ):
        images, metadata, mask = load_test_study(study_uid)
        logits = model(
            images.to(device),
            metadata.to(device),
            mask.to(device),
        )
        predictions.append(logits.sigmoid().cpu().numpy()[0])

predictions = np.stack(predictions)

assert predictions.shape == (len(test), len(TARGETS))
assert np.isfinite(predictions).all(), "Non-finite predictions detected."
assert ((predictions >= 0) & (predictions <= 1)).all()
assert test["StudyInstanceUID"].notna().all()
assert test["StudyInstanceUID"].is_unique

submission = test[["StudyInstanceUID"]].copy()
submission[TARGETS] = predictions
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission shape:", submission.shape)
print("Prediction range:", float(predictions.min()), "to", float(predictions.max()))
print("Unreadable sampled slices:", load_counts["unreadable_slices"])
print("Series with no readable sampled slices:", load_counts["empty_series"])
print("Saved: submission.csv")